# 03 Experimente ausführen

Dieses Notebook führt die drei Experimente aus. Jedes Experiment trainiert alle fünf Varianten (A-E) mit je 20 Seeds, also 100 Läufe. Pro Seed starten alle Varianten mit identischen Gewichten (gepaartes Design).

Experiment 2 und 3 ändern gegenüber Experiment 1 jeweils genau eine Sache, damit sich ein Unterschied im Ergebnis eindeutig zuordnen lässt.

Die Konfiguration steht in `EXPERIMENTS` in `experiment.py`, das Training in `run_experiment`. Jeder Lauf wird als eigene CSV-Datei in `results/<experiment>/runs/` gespeichert (`run_path`). Läufe, deren Datei schon existiert, werden übersprungen: Ein Experiment kann also abgebrochen und später fortgesetzt werden. Um ein Experiment komplett neu zu rechnen, den Ordner `results/<experiment>/runs/` löschen.

Die Zellen, die Experimente starten, laufen je nach Experiment zwischen 20 Minuten und 3 Stunden (auf einer RTX 5070 Ti) und sollten einzeln von Hand ausgeführt werden. Schneller geht es im Terminal, wo sich ein Experiment auf mehrere parallele Prozesse verteilen lässt (siehe Experiment 2). Die Auswertung folgt in Notebook 04.

In [1]:
import time

import pandas as pd
import torch

from batch_shuffling_regularization.experiment import EXPERIMENT_VARIANTS, EXPERIMENTS, SEEDS, Experiment, load_results, run_experiment, run_path
from batch_shuffling_regularization.training import make_deterministic

make_deterministic()
device = torch.device("cuda")
print(torch.cuda.get_device_name(device))

NVIDIA GeForce RTX 5070 Ti


## Konfiguration

Konstant über alle Shuffle-Varianten eines Experiments: Modell (784-256-256-10, ReLU), Initialisierung pro Seed, Optimierer (plain SGD), Lernrate, Batchgröße, Epochen und Daten-Split (50.000 Train / 10.000 Validation / 10.000 Test). Nur die Reihenfolge der Trainingsdaten unterscheidet sich.

In [2]:
pd.DataFrame(
    [{"Variante": v.name, "Batch-Order-Funktion": v.batch_order.__name__, "nach Klasse sortiert": v.sort_by_class} for v in EXPERIMENT_VARIANTS]
).set_index("Variante")

,Batch-Order-Funktion,nach Klasse sortiert
Variante,,
A,no_shuffle,False
B,shuffle_batch_order,False
C,shuffle_within_batches,False
D,full_reshuffle,False
E,no_shuffle,True


In [3]:
print("Seeds:", list(SEEDS))
est_runtimes = [20, 180, 20]
pd.DataFrame(
    [
        {
            "Experiment": e.name,
            "Datensatz": e.dataset,
            "Epochen": e.cfg.epochs,
            "Batchgröße": e.cfg.batch_size,
            "Lernrate": e.cfg.lr,
            "Schritte pro Epoche": 50_000 // e.cfg.batch_size,
            "Gesch. Durchlaufzeit in min": t
        }
        for e, t in zip(EXPERIMENTS.values(), est_runtimes)
    ]
).set_index("Experiment")

Seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


,Datensatz,Epochen,Batchgröße,Lernrate,Schritte pro Epoche,Gesch. Durchlaufzeit in min
Experiment,,,,,,
mnist_bs100,mnist,30,100,0.10,500,20
mnist_bs10,mnist,30,10,0.01,5000,180
fashion_bs100,fashion_mnist,30,100,0.10,500,20


## Stand aller Experimente

Anzahl fertiger Läufe pro Experiment und Variante (Ziel: 20 pro Zelle).

In [4]:
def run_status(experiment: Experiment) -> pd.DataFrame:
    return pd.DataFrame(
        {f"Seed {seed}": ["✓" if run_path(experiment, v.name, seed).exists() else "" for v in EXPERIMENT_VARIANTS] for seed in SEEDS},
        index=[v.name for v in EXPERIMENT_VARIANTS],
    )


pd.DataFrame(
    {name: (run_status(e) == "✓").sum(axis=1) for name, e in EXPERIMENTS.items()}
).T

,A,B,C,D,E
mnist_bs100,20,20,20,20,20
mnist_bs10,20,20,20,20,20
fashion_bs100,20,20,20,20,20


## Experiment 1: `mnist_bs100` (Referenz)

MNIST, Batchgröße 100, Lernrate 0,1, 30 Epochen. Das ist das ursprüngliche Setup mit 20 Seeds.

Laufzeit: etwa 11 s pro Lauf, also rund 20 min für alle 100 Läufe.

In [5]:
start = time.perf_counter()
run_experiment(EXPERIMENTS["mnist_bs100"], device)
print(f"Fertig nach {(time.perf_counter() - start) / 60:.1f} min")
run_status(EXPERIMENTS["mnist_bs100"])

Fertig nach 0.0 min


,Seed 0,Seed 1,Seed 2,Seed 3,Seed 4,Seed 5,Seed 6,Seed 7,Seed 8,Seed 9,Seed 10,Seed 11,Seed 12,Seed 13,Seed 14,Seed 15,Seed 16,Seed 17,Seed 18,Seed 19
A,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓
B,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓
C,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓
D,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓
E,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓


## Experiment 2: `mnist_bs10` (kleine Batches)

Wie Experiment 1, aber mit Batchgröße 10 statt 100.

**Warum:** Bei Batchgröße 100 enthält jeder Batch fast alle 10 Ziffern. Die festen Batches von B sind also schon repräsentativ, und die neuen Batch-Kombinationen von D bringen kaum neue Information. Bei Batchgröße 10 enthält ein Batch nur etwa 6-7 verschiedene Ziffern, und bei B landen dieselben 10 Bilder in jeder Epoche zusammen. Wenn die Zusammensetzung der Batches eine Rolle spielt, sollte sich D hier von B absetzen.

**Lernrate 0,01 statt 0,1:** Mit 10-mal kleineren Batches macht das Modell 10-mal so viele Schritte pro Epoche (5.000 statt 500). Damit der Fortschritt pro Epoche vergleichbar bleibt und das Training nicht instabiler wird, wird die Lernrate um denselben Faktor verkleinert (lineare Skalierung: Lernrate / Batchgröße bleibt konstant). Batchgröße und Lernrate ändern sich hier also gemeinsam; einzeln wären sie nicht sinnvoll vergleichbar.

Laufzeit: etwa 105 s pro Lauf, also rund **180 Minuten** für alle 100 Läufe.

**Schneller im Terminal:** Bei Batchgröße 10 ist nicht die Rechenleistung der GPU der Engpass, sondern der Overhead pro Schritt (Python und Kernel-Starts): Ein einzelner Prozess lastet die GPU nur zu einem Teil aus. Deshalb lässt sich ein Experiment auf mehrere Prozesse aufteilen, die parallel laufen. `--shard i/n` rechnet nur die Seeds mit `seed % n == i` (`parse_args` in `experiment.py`), zum Beispiel in vier Terminals:

```
uv run python -m batch_shuffling_regularization.experiment mnist_bs10 --shard 0/4
uv run python -m batch_shuffling_regularization.experiment mnist_bs10 --shard 1/4
uv run python -m batch_shuffling_regularization.experiment mnist_bs10 --shard 2/4
uv run python -m batch_shuffling_regularization.experiment mnist_bs10 --shard 3/4
```

Die Ergebnisse sind dieselben wie mit einem einzelnen Prozess, weil jeder Lauf deterministisch ist und nur von seinem Seed abhängt. Fertige Läufe werden auch hier übersprungen. Das funktioniert genauso für die anderen Experimente; ohne Experimentnamen laufen alle drei nacheinander.

In [6]:
start = time.perf_counter()
run_experiment(EXPERIMENTS["mnist_bs10"], device)
print(f"Fertig nach {(time.perf_counter() - start) / 60:.1f} min")
run_status(EXPERIMENTS["mnist_bs10"])

Fertig nach 0.0 min


,Seed 0,Seed 1,Seed 2,Seed 3,Seed 4,Seed 5,Seed 6,Seed 7,Seed 8,Seed 9,Seed 10,Seed 11,Seed 12,Seed 13,Seed 14,Seed 15,Seed 16,Seed 17,Seed 18,Seed 19
A,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓
B,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓
C,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓
D,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓
E,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓


## Experiment 3: `fashion_bs100` (schwierigerer Datensatz)

Wie Experiment 1, aber mit Fashion-MNIST statt MNIST: 10 Klassen von Kleidungsstücken (T-Shirt, Hose, Pullover, Kleid, Mantel, Sandale, Hemd, Sneaker, Tasche, Stiefel) im selben Format (28×28 Graustufen, 60.000 Train / 10.000 Test). Geladen wird es mit `load_dataset(device, "fashion_mnist")` in `data.py`.

**Warum:** Fashion-MNIST ist schwerer zu trennen, das Netz overfittet deutlich stärker und der Generalization Gap ist größer. Regularisierende Effekte haben dadurch mehr Raum, sichtbar zu werden.

**Hyperparameter wie Experiment 1** (Batchgröße 100, Lernrate 0,1, 30 Epochen), damit nur der Datensatz wechselt.

Laufzeit: etwa 12 s pro Lauf, also rund 20 min für alle 100 Läufe.

In [7]:
start = time.perf_counter()
run_experiment(EXPERIMENTS["fashion_bs100"], device)
print(f"Fertig nach {(time.perf_counter() - start) / 60:.1f} min")
run_status(EXPERIMENTS["fashion_bs100"])

Fertig nach 0.0 min


,Seed 0,Seed 1,Seed 2,Seed 3,Seed 4,Seed 5,Seed 6,Seed 7,Seed 8,Seed 9,Seed 10,Seed 11,Seed 12,Seed 13,Seed 14,Seed 15,Seed 16,Seed 17,Seed 18,Seed 19
A,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓
B,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓
C,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓
D,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓
E,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓


## Vollständigkeit

Pro Experiment und Variante sollten 20 Seeds × 31 Epochen (0-30) vorliegen, ohne fehlende Werte. `load_results` lädt nur fertige Läufe, deshalb hier die Kontrolle vor der Auswertung.

In [8]:
pd.DataFrame(
    [
        {
            "Experiment": name,
            "Variante": variant,
            "Seeds": group["seed"].nunique(),
            "Epochen": group["epoch"].nunique(),
            "fehlende Werte": int(group.isna().sum().sum()),
        }
        for name, e in EXPERIMENTS.items()
        if run_status(e).eq("✓").any().any()
        for variant, group in load_results(e).groupby("variant")
    ]
).set_index(["Experiment", "Variante"])

Seeds  Epochen  fehlende Werte
Experiment    Variante                                
mnist_bs100   A            20       31               0
              B            20       31               0
              C            20       31               0
              D            20       31               0
              E            20       31               0
mnist_bs10    A            20       31               0
              B            20       31               0
              C            20       31               0
              D            20       31               0
              E            20       31               0
fashion_bs100 A            20       31               0
              B            20       31               0
              C            20       31               0
              D            20       31               0
              E            20       31               0